**Autores**

* DIANA CAROLINA LEAL MARTINEZ
* DIEGO ALEJANDRO FIGUEREDO SALAMANCA
* MARIA CAMILA REYES GIL
* SILVIA CARDENAS SANTOS

## 1. Conexión a la base de datos
En el siguiente código se configuran las conexiones JDBC para extraer la información desde la base de datos transaccional origen y cargarla en el esquema de destino personal (`_ELT`).

In [ ]:
# Configuración servidor base de datos transaccional
db_user = 'DB_202615_'
db_psswd = ''  
db_host = '157.253.236.120'
db_port = 8080
db_destino = 'DB_202615_' 

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'
source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/WWImportersTransactional?{jdbc_opciones}'
dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [2]:
import os
import sys
from pathlib import Path
import urllib.request

from pyspark.sql import SparkSession, functions as sf
from pyspark.sql.functions import col, when, regexp_replace

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))

if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    try:
        (hadoop_home / 'bin' / 'winutils.exe').touch(exist_ok=True)
    except:
        pass
    os.environ['HADOOP_HOME'] = str(hadoop_home)

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-ETL-AreaGeografia-MariaCamila')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print("¡Spark inicializado correctamente con el conector JDBC!")

¡Spark inicializado correctamente con el conector JDBC!


In [3]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

## 2. Extracción de áreas de servicio
Extracción de los datos transaccionales desde la fuente `FuenteAreasDeServicio_ETL`.

In [4]:
sql_fuente_areas_servicios = """
(SELECT 
    IdAreaDeServicio_T,
    NombreAreaDeServicio,
    IdGeografia_T,
    Condado,
    Estado,
    PoblacionAct,
    Area,
    Densidad,
    Fecha
FROM
    RaSaTransaccional_ETL.FuenteAreasDeServicio_ETL
WHERE IdAreaDeServicio_T IS NOT NULL AND IdGeografia_T IS NOT NULL) as area_servicio
"""

df_areas_servicio_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_areas_servicios, db_user, db_psswd)
df_areas_servicio_raw.show(10)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|      464269|805.0|   577.0| 2017|
|         100722019|New Jersey  - Med...|        34023| Middlesex County|New Jersey|      860807|311.0|  2768.0| 2019|
|         100922020|New Jersey - Medi...|        34019| Hunterdon County|New Jersey|      129924|430.0|   302.0| 2020|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2018|
|         101062020|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2020|
|         101082018|New Jersey - Medi...|       

## 3. Transformación y reglas de negocio
Aplicación de reglas de negocio de RaSA:
- Corrección de áreas negativas multiplicando por `-1`.
- Corrección del error de digitación en población actual eliminando los últimos 4 dígitos (`0001`).
- Construcción de las dimensiones `Geografia`, `AreasDeServicio` y la tabla de asociación `AsociacionAreaServicioGeografia`.

In [5]:
from pyspark.sql import functions as sf

# 1. Limpieza y aplicación de reglas de negocio
df_as_limpio = df_areas_servicio_raw \
    .withColumn("Area_Corregido", sf.when(sf.col("Area") < 0, sf.col("Area") * -1).otherwise(sf.col("Area"))) \
    .withColumn(
        "PoblacionAct_Corregido",
        sf.when(sf.col("PoblacionAct").cast("string").rlike("0001$"), 
                sf.regexp_replace(sf.col("PoblacionAct").cast("string"), "0001$", "").cast("long"))
        .otherwise(sf.col("PoblacionAct"))
    )

# 2. Dimensión GEOGRAFIA
df_geografia_base = df_as_limpio.select(
    "IdGeografia_T", "Estado", "Condado", "PoblacionAct_Corregido", "Area_Corregido", "Densidad"
).dropDuplicates(["IdGeografia_T"])

df_geografia = df_geografia_base.coalesce(1).withColumn("IdGeografia_DWH", sf.monotonically_increasing_id() + 1)

df_geografia_final = df_geografia.select(
    "IdGeografia_DWH",
    "IdGeografia_T",
    sf.col("Estado").alias("Estado"),
    sf.col("Condado").alias("Condado"),
    sf.col("Area_Corregido").alias("AreaAct"),
    sf.col("Densidad").alias("DensidadAct"),
    sf.col("PoblacionAct_Corregido").alias("PoblacionAct")
)

# 3. Dimensión AREAS DE SERVICIO
df_areas_base = df_as_limpio.select(
    "IdAreaDeServicio_T", 
    sf.col("NombreAreaDeServicio").alias("Nombre"), 
    sf.col("Fecha").alias("AnnioCreacion")
).dropDuplicates(["IdAreaDeServicio_T"])

df_areas_servicio = df_areas_base.coalesce(1).withColumn("IdAreaDeServicio_DWH", sf.monotonically_increasing_id() + 1)

df_areas_servicio_final = df_areas_servicio.select(
    "IdAreaDeServicio_DWH",
    "IdAreaDeServicio_T",
    "Nombre",
    "AnnioCreacion"
)

# 4. Tabla de Asociación (AsociacionAreaServicioGeografia)
df_asociacion = df_as_limpio.join(
    df_areas_servicio_final, df_as_limpio["IdAreaDeServicio_T"] == df_areas_servicio_final["IdAreaDeServicio_T"], "inner"
).join(
    df_geografia_final, df_as_limpio["IdGeografia_T"] == df_geografia_final["IdGeografia_T"], "inner"
).select(
    "IdAreaDeServicio_DWH",
    "IdGeografia_DWH"
).dropDuplicates()

## 4. Carga de datos al esquema DWH
Persistencia de los resultados utilizando el sufijo `_ELT`.

In [6]:
# Cargar Geografía
guardar_db(dest_db_connection_string, df_geografia_final, f'{db_destino}.Geografia_ELT', db_user, db_psswd)

# Cargar Áreas de Servicio
guardar_db(dest_db_connection_string, df_areas_servicio_final, f'{db_destino}.AreasDeServicio_ELT', db_user, db_psswd)

# Cargar Asociación
guardar_db(dest_db_connection_string, df_asociacion, f'{db_destino}.AsociacionAreaServicioGeografia_ELT', db_user, db_psswd)

print("¡Proceso ETL de Geografía, Áreas de Servicio y Asociación finalizado y cargado con éxito!")

¡Proceso ETL de Geografía, Áreas de Servicio y Asociación finalizado y cargado con éxito!
